# Feature Engineering - ATP Tennis Match Predictor

This notebook builds the feature set used to predict ATP match winners based on findings from the EDA notebook ('01_eda.ipynb'). Rather than using raw match data directly, this notebook transforms it into features that describe the *relative* gap between two players: their ranking, points, and historical performance, since that's what actually determines who's more likely to win a given match.

**Key decisions carried over from EDA:**
- 'Pts_1'/'Pts_2' use '-1' as a missing data placeholder in ~23% of rows (not real NaN)
- 'Odd_1'/'Odd_2' are only reliably available from ~2005 onward
- 'rank_diff' (corr = -0.24) and 'points_diff' (corr = 0.32) both showed real relationships with match outcome and are treated as core features

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/processed/atp_matches_clean.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.shape

(68274, 21)

CSVs don't preserve datetime types, so 'Date' needs to be re-converted with 'pd.to_datetime()' every time the file is reloaded.

In [4]:
df['points_data_missing'] = ((df['Pts_1'] == -1) | (df['Pts_2'] == -1)).astype(int)

df['points_diff'] = np.where(
    df['points_data_missing'] == 1,
    0,
    df['Pts_1'] - df['Pts_2']
)

df['points_data_missing'].mean()

np.float64(0.22888654539063186)

'points_data_missing' is a separate binary column (1 = we don't know the points gap) that the model can learn to use as a signal, letting it learn to trust 'points_diff' less whenever this flag is on.